<a href="https://colab.research.google.com/github/TrajanDS/SFT_Agent_POC/blob/main/data_gathering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import pandas as pd
import time
from google.colab import drive

In [ ]:
CFPB_API_URL = "https://www.consumerfinance.gov/data-research/consumer-complaints/search/api/v1/"

def fetch_cfpb_complaints(
    max_records=10_000,
    batch_size=1_000,
    date_received_min="2024-01-01",
    date_received_max="2026-01-01",
    has_narrative=True,
):
    """
    Purpose:
    Retrieves complaint data from the Consumer Financial Protection Bureau's
     API for Citizen Financial Group.

    ------------------------------------------------------------------------
    Parameters:
    max_records (int): Maximum number of records to fetch.
    batch_size (int): Number of records to fetch in each API call.

    ------------------------------------------------------------------------
    TODO:
      * Randomize time period being drawn from
      * Get set number of samples for each type
    """


    rows = []
    offset = 0
    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; CFPB-DataFetcher/1.0; https://your-site-or-email)"
    }

    while len(rows) < max_records:
        current_batch = min(batch_size, max_records - len(rows))

        params = {
            "frm": offset,
            "size": current_batch,
            "no_aggs": "true",
            "company":"CITIZENS FINANCIAL GROUP, INC.",
            "date_received_min": date_received_min,
        }
        if date_received_max:
            params["date_received_max"] = date_received_max
        if has_narrative:
            params["has_narrative"] = "true"

        try:
            response = requests.get(CFPB_API_URL, params=params, headers=headers, timeout=60)
            response.raise_for_status()
            data = response.json()

            hits = data.get("hits", {}).get("hits", [])
            if not hits:
                print("No more results returned.")
                break

            # Extract the actual complaint data (_source)
            for hit in hits:
                if "_source" in hit:
                    rows.append(hit["_source"])
                else:
                    rows.append(hit)

            print(f"Fetched {len(hits)} records (total so far: {len(rows)})")

            offset += len(hits)

            # Be polite to the API
            if len(hits) < current_batch:
                break  # last page

            time.sleep(0.5)  # small delay to avoid rate-limiting

        except requests.exceptions.RequestException as e:
            print(f"Error fetching data: {e}")
            break

    print(f"Completed: {len(rows)} records fetched.")
    return rows


# Call API
rows = fetch_cfpb_complaints()

# Form rows from API as dataframe
df = pd.DataFrame(rows)

In [ ]:
df.company.value_counts()

In [ ]:
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

In [ ]:
drive.mount('/content/drive', force_remount=True)

In [ ]:
tdf = pd.DataFrame(fetch_cfpb_complaints(
    max_records=1_000,
    batch_size=1_000,
    date_received_min="2024-01-01",
    date_received_max="2024-01-05",
    has_narrative=True,
)
)

In [ ]:
tdf.complaint_what_happened.values[0:4]

In [ ]:
tdf.complaint_what_happened.values[0:4]